In [5]:
import yaml
import os
import re

def detect_bugs_in_yaml(yaml_dir):
    """
    Scans YAML files for discrepancies between the correct_answer 
    and the 3rd item in the explanation sequence (the one after the 2nd '->').
    """
    if not os.path.isdir(yaml_dir):
        print(f"Directory not found: {yaml_dir}")
        return

    bug_count = 0

    for filename in os.listdir(yaml_dir):
        if filename.endswith(('.yaml', '.yml')):
            yaml_path = os.path.join(yaml_dir, filename)
            
            with open(yaml_path, 'r', encoding='utf-8') as f:
                try:
                    yaml_data = yaml.safe_load(f)
                except Exception as e:
                    print(f"Error reading {filename}: {e}")
                    continue
                
                # Handle both lists of questions or dictionaries containing a 'questions' list
                questions_list = []
                if isinstance(yaml_data, list):
                    questions_list = yaml_data
                elif isinstance(yaml_data, dict):
                    questions_list = yaml_data.get('questions', [])
                
                for entry in questions_list:
                    if not isinstance(entry, dict):
                        continue
                        
                    q_num = entry.get('question_number', 'Unknown')
                    correct_answer = str(entry.get('correct_answer', '')).strip()
                    explanation = entry.get('explanation', '')
                    
                    if not explanation:
                        continue
                    
                    # Pattern explanation format: "順序: 2 (今週の) -> 1 (予定が) -> 4 (詳しく) -> 3 (書いて)。★(3番目)に入るのは「3」です。"
                    # We want to look at the part after the 2nd "->" (which corresponds to the 3rd position/star position)
                    # Let's split by "->"
                    parts = explanation.split('->')
                    
                    if len(parts) >= 3:
                        # The third segment contains the 3rd item, e.g., " 4 (詳しく) "
                        third_segment = parts[2]
                        
                        # Extract the option number right before the parenthesis or whitespace
                        match = re.search(r'(\d+)\s*\(', third_segment)
                        if match:
                            explanation_star_answer = match.group(1)
                            
                            # Compare with correct_answer
                            if explanation_star_answer != correct_answer:
                                bug_count += 1
                                print(f"[{filename}] Question #{q_num} Bug Detected:")
                                print(f"  - correct_answer field : '{correct_answer}'")
                                print(f"  - Explanation specifies: '{explanation_star_answer}' (from sequence: {explanation.strip()})")
                                print("-" * 60)

    print(f"\nScan complete. Total potential bugs found: {bug_count}")

# --- Example Usage ---
if __name__ == "__main__":
    detect_bugs_in_yaml(yaml_dir=r"C:\Users\User\Documents\Skillmap\japanese\tests")


Scan complete. Total potential bugs found: 0
